# Practice: Choose a Model for a Real Project

**Computational Text Analysis Pack · Module 26 — Fine-Tuning Pretrained Models**

*Technique:* A costed comparison across four workflows on one task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/13-african-nlp-models/26c_practice_choose_your_model.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **You have 300 hand-labelled documents and 20,000 unlabelled ones. Which
> workflow should you use, and what does each cost?**

This is the most common shape a real DH project takes, and the answer is rarely
the most sophisticated option.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
BACKEND, _pod, _why = "cached", None, ""
if IN_COLAB:
    try:
        import transformers  # noqa: F401
    except ImportError:
        os.system("pip install -q transformers")
    import torch
    BACKEND = "colab-gpu" if torch.cuda.is_available() else "colab-cpu"
else:
    try:
        sys.path.insert(0, "/app")
        from utils import gpu_pod as _pod
        _pod.health(); BACKEND = "coursehub-pod"
    except Exception as exc:
        _why = str(exc)[:70]
POD_LIVE = BACKEND != "cached"
print(f"backend: {BACKEND}" + (f"  ({_why})" if _why else ""))

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np, time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# The realistic setting: a SMALL labelled set.
N_LABELLED = 300
sub = df[df["n_words"] >= 30].sample(n=N_LABELLED, random_state=3).reset_index(drop=True)
tr, te = train_test_split(np.arange(len(sub)), test_size=0.35, random_state=0,
                          stratify=sub["category"])
print(f"{len(tr)} training / {len(te)} test documents, "
      f"{sub['category'].nunique()} classes, {sub['language'].nunique()} languages")
gold = sub["category"].iloc[te]
results = []

In [ ]:
# Workflow 1: Naive Bayes on counts. The cheapest thing that could work.
t0 = time.time()
v = TfidfVectorizer(token_pattern=r"\w+", min_df=1, max_features=20000)
A, B = v.fit_transform(sub["text"].iloc[tr]), v.transform(sub["text"].iloc[te])
nb = MultinomialNB().fit(A, sub["category"].iloc[tr])
results.append({"workflow": "Naive Bayes + TF-IDF", "acc": accuracy_score(gold, nb.predict(B)),
                "secs": time.time()-t0, "gpu": "no", "labels_needed": "any"})

# Workflow 2: logistic regression. Still cheap, usually better.
t0 = time.time()
lr = LogisticRegression(max_iter=1000).fit(A, sub["category"].iloc[tr])
results.append({"workflow": "Logistic + TF-IDF", "acc": accuracy_score(gold, lr.predict(B)),
                "secs": time.time()-t0, "gpu": "no", "labels_needed": "hundreds"})
for r in results:
    print(f"{r['workflow']:<26}{r['acc']:>7.3f}{r['secs']:>8.1f}s")

In [ ]:
# Workflow 3: frozen pretrained embeddings + the same classifier.
bank = np.load("labse_headlines.npy")
index = pd.read_csv("labse_headlines_index.csv")
ok = index["category"].notna().values
Xe, ye = bank[ok], index.loc[ok, "category"].values
# Match the small-data setting: only 300 labelled.
idx = np.random.default_rng(0).permutation(len(Xe))[:N_LABELLED]
Xs, ys = Xe[idx], ye[idx]
etr, ete = train_test_split(np.arange(len(Xs)), test_size=0.35, random_state=0, stratify=ys)
t0 = time.time()
em = LogisticRegression(max_iter=2000).fit(Xs[etr], ys[etr])
results.append({"workflow": "Frozen LaBSE + logistic",
                "acc": accuracy_score(ys[ete], em.predict(Xs[ete])),
                "secs": time.time()-t0, "gpu": "once, to embed", "labels_needed": "hundreds"})
print(f"{results[-1]['workflow']:<26}{results[-1]['acc']:>7.3f}"
      f"{results[-1]['secs']:>8.1f}s")
print()
print("Note this row used headlines and the first two used full text, so it is")
print("not a clean head-to-head — module 23d builds the fair version. Read it as")
print("a workflow comparison, not a leaderboard.")

In [ ]:
# Workflow 4: fine-tuning. Reported from module 26a rather than re-run here.
results.append({"workflow": "Fine-tuned transformer", "acc": float("nan"),
                "secs": float("nan"), "gpu": "yes, hours", "labels_needed": "thousands"})
tab = pd.DataFrame(results)
print(tab.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("The labels_needed column is the one that decides this project. With 300")
print("labelled documents, fine-tuning is not an option that is merely expensive —")
print("it is an option that will overfit and underperform the cheap rows.")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
done = tab.dropna(subset=["acc"])
ax.barh(done["workflow"], done["acc"], color="steelblue")
ax.set_xlabel("accuracy"); ax.set_title(f"Four workflows, {N_LABELLED} labelled documents")
plt.tight_layout()

## The decision, written out

With **300 labels and 20,000 unlabelled documents**:

- **Use logistic regression on TF-IDF.** It is minutes, interpretable, and at
  this label count competitive with anything else.
- **Add frozen embeddings if the corpus is multilingual**, because that is where
  transfer earns its cost (module 26b).
- **Do not fine-tune.** You do not have the labels for it.
- **Then label more.** The single highest-return action in this project is not a
  better model; it is another 300 labels. Module 17's agreement protocol is how
  you get them without producing noise.

**Defensible:** "With 300 labelled documents we compared four workflows;
logistic regression on TF-IDF achieved [X] and was selected for
interpretability and cost. Fine-tuning was not attempted at this label count."

### 🔧 Your turn 1

Rerun with N_LABELLED at 100 and at 1,000. At what point does the ranking change?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The cheap methods dominate at 100 and stay competitive far longer than most
people expect; the pretrained representation pulls ahead when labels are scarce
relative to the number of classes, which is the transfer effect doing exactly
what it is supposed to. The crossover is corpus-specific, which is why the
answer to "which model should I use" is always "run this comparison on your
data" and never a recommendation from a paper about someone else's.

</details>